# Contas da janela

Números **reais**, coletados ao vivo com o **app17-3** (100 Hz, valores em **g**), nas 4
situações que viram as classes do app17-7:

| Situação | O que foi feito |
|---|---|
| `operando` | sensor parado e nivelado |
| `inclinado_frente` | inclinado para a frente |
| `inclinado_tras` | inclinado para trás |
| `anomalia` | chacoalhando |

A saída original do Serial de cada coleta está na pasta `coleta_professor/` do app17-4.

1. As contas do ESP32, em Python.
2. Cinco leituras de cada situação, feitas à mão.
3. As 4 situações lado a lado.
4. O que o ESP32 calcula com as 100 leituras.
5. Divide por *n* ou por *n − 1*?

## 1) As contas do ESP32, em Python

As mesmas contas do **app17-4** (e do app17-7), com os mesmos nomes de função, escritas
com o que o Python já oferece: `sum`, `len`, `max`, `min` e o módulo `statistics`. A conta é
a mesma do firmware; só a escrita é mais curta.

In [ ]:
import math
import statistics
import pandas as pd

def calcMean(arr):                  # média: para onde a gravidade aponta?
    return sum(arr) / len(arr)

def calcStd(arr):                   # desvio: quanto o eixo treme?
    return statistics.pstdev(arr)   # "p" de população: divide por n, como o ESP32

def calcMagnitude(ax, ay, az):      # magnitude de cada leitura: os 3 eixos juntos
    mag = []
    for x, y, z in zip(ax, ay, az):
        mag.append(math.sqrt(x**2 + y**2 + z**2))
    return mag

def calcPtP(arr):                   # amplitude total (pico a pico): o maior tranco
    return max(arr) - min(arr)

> **Sobre o nome:** `calcPtP` e `p2p_mag` vêm de *peak-to-peak*, **pico a pico**, o nome
> usado na engenharia de sinais. Em estatística, máximo − mínimo se chama **amplitude
> total**. Cuidado: em física, a "amplitude" de uma onda é só a **metade** disso (do centro
> até o pico).

E o bloco que, no `loop()` do app17-4, transforma a janela nas **8 features**:

In [ ]:
def calcFeatures(ax, ay, az):
    mag = calcMagnitude(ax, ay, az)
    return {
        "mean_ax": calcMean(ax), "mean_ay": calcMean(ay), "mean_az": calcMean(az),
        "std_ax": calcStd(ax), "std_ay": calcStd(ay), "std_az": calcStd(az),
        "std_mag": calcStd(mag), "p2p_mag": calcPtP(mag),
    }

resultado = {}   # aqui guardamos as 8 features de cada situação

## 2) Cinco leituras de cada situação

- **Parado e inclinado:** 5 leituras **seguidas**. O sinal quase não muda.
- **Chacoalhando:** **uma leitura a cada 10** (a 1ª, 11ª, 21ª, 31ª e 41ª da janela). Cinco
  seguidas a 100 Hz cobririam só 40 ms, um pedaço de um único movimento.

> Na aula, troque estes números pelos que forem coletados ao vivo.

### `operando` — parado e nivelado

In [ ]:
ax = [-0.004, 0.000, -0.003, -0.002, 0.002]
ay = [0.001, 0.002, -0.002, 0.000, 0.001]
az = [0.996, 0.995, 0.995, 0.997, 0.999]

mag = calcMagnitude(ax, ay, az)
display(pd.DataFrame({"ax": ax, "ay": ay, "az": az, "magnitude": mag}).round(3))
print("média de az =", round(calcMean(az), 3), "g  -> a gravidade está toda no eixo z")
resultado["operando"] = calcFeatures(ax, ay, az)

### `inclinado_frente`

In [ ]:
ax = [0.136, 0.132, 0.133, 0.132, 0.130]
ay = [0.468, 0.465, 0.465, 0.465, 0.462]
az = [0.845, 0.844, 0.843, 0.841, 0.841]

mag = calcMagnitude(ax, ay, az)
display(pd.DataFrame({"ax": ax, "ay": ay, "az": az, "magnitude": mag}).round(3))
print("média de ay =", round(calcMean(ay), 3), "g  -> parte da gravidade foi para o eixo y")
print("média de az =", round(calcMean(az), 3), "g")
resultado["inclinado_frente"] = calcFeatures(ax, ay, az)

### `inclinado_tras`

In [ ]:
ax = [-0.086, -0.087, -0.088, -0.083, -0.083]
ay = [-0.629, -0.630, -0.626, -0.626, -0.625]
az = [0.821, 0.817, 0.819, 0.824, 0.830]

mag = calcMagnitude(ax, ay, az)
display(pd.DataFrame({"ax": ax, "ay": ay, "az": az, "magnitude": mag}).round(3))
print("média de ay =", round(calcMean(ay), 3), "g  -> agora para o outro lado")
print("média de az =", round(calcMean(az), 3), "g")
resultado["inclinado_tras"] = calcFeatures(ax, ay, az)

Inclinar **não muda a magnitude**: ela continua perto de 1 g, porque é a gravidade. O que
muda é **para qual eixo** a gravidade aponta. Regra de bolso: inclinar ~30° leva ~0,5 g para
outro eixo; 45°, ~0,7 g; 90°, 1 g.

Neste sensor, preso deste jeito, a gravidade foi para o eixo **y**. Em outra montagem pode
ser o x: depende de como o sensor está preso.

### `anomalia` — chacoalhando

In [ ]:
ax = [0.418, 0.329, 0.118, -0.049, 0.515]
ay = [0.274, 0.296, -0.105, 0.028, -0.111]
az = [3.945, 1.688, -0.468, -1.398, 1.315]

resultado["anomalia"] = calcFeatures(ax, ay, az)

O desvio de `az`, em 4 passos:

1. a **diferença** de cada leitura até a média;
2. **ao quadrado** (some o sinal de menos);
3. a **média** dos quadrados, dividindo por *n*, como o ESP32;
4. a **raiz**, para voltar à unidade g.

In [ ]:
media_az = calcMean(az)

diferenca = []
quadrado = []
for v in az:
    d = v - media_az
    diferenca.append(d)        # passo 1: diferença até a média
    quadrado.append(d ** 2)    # passo 2: ao quadrado

display(pd.DataFrame({"az": az, "1) diferença": diferenca, "2) ao quadrado": quadrado}).round(3))

media_dos_quadrados = sum(quadrado) / len(quadrado)   # passo 3: média (divide por n)
desvio = math.sqrt(media_dos_quadrados)               # passo 4: raiz

print("média de az            =", round(media_az, 3), "g  -> a gravidade continua lá")
print("3) média dos quadrados =", round(media_dos_quadrados, 3))
print("4) desvio = raiz       =", round(desvio, 3), "g  -> o tamanho do balanço")
print("   calcStd(az)         =", round(calcStd(az), 3), "g  -> o mesmo número")

Chacoalhando, a **média de z continua perto de 1 g**: para cima e para baixo, os valores se
compensam e sobra a gravidade. O **desvio não se compensa**: ele mede o tamanho do balanço.
É a feature de vibração.

> O desvio é o RMS depois de tirar a média: sai a parte parada (a gravidade), sobra a
> vibração.

## 3) As 4 situações lado a lado

In [ ]:
tabela = pd.DataFrame(resultado).T.round(3)
tabela

- **A média separa as inclinações:** `mean_ay` é positivo na frente e negativo atrás. Mas
  `operando` e `anomalia` têm a mesma média de z (~1 g): **a média não separa esses dois**.
- **O desvio e a amplitude total separam a anomalia:** centenas de vezes maiores. Mas frente e
  trás têm o mesmo desvio: **o desvio não separa esses dois**.

Nenhuma família sozinha separa as 4 situações. As 8 juntas, sim: é por isso que o app17-7
envia as 8.

## 4) E o ESP32, com as 100 leituras?

O app17-4 faz exatamente estas contas, mas com a **janela inteira** (100 leituras). Estes são
os números dele para as mesmas janelas de onde saíram as 5 leituras:

In [ ]:
esp32 = {
    "operando": {"mean_ax": 0.000, "mean_ay": 0.001, "mean_az": 0.998, "std_ax": 0.002, "std_ay": 0.001, "std_az": 0.002, "std_mag": 0.002, "p2p_mag": 0.010},
    "inclinado_frente": {"mean_ax": 0.132, "mean_ay": 0.466, "mean_az": 0.844, "std_ax": 0.002, "std_ay": 0.002, "std_az": 0.004, "std_mag": 0.003, "p2p_mag": 0.015},
    "inclinado_tras": {"mean_ax": -0.089, "mean_ay": -0.627, "mean_az": 0.824, "std_ax": 0.005, "std_ay": 0.002, "std_az": 0.006, "std_mag": 0.004, "p2p_mag": 0.024},
    "anomalia": {"mean_ax": 0.177, "mean_ay": 0.173, "mean_az": 0.965, "std_ax": 0.275, "std_ay": 0.256, "std_az": 1.863, "std_mag": 1.271, "p2p_mag": 4.417},
}

esp32 = pd.DataFrame(esp32).T
esp32

In [ ]:
comparacao = pd.DataFrame({
    "mean_ay (5 leituras)": tabela["mean_ay"],
    "mean_ay (100, ESP32)": esp32["mean_ay"],
    "std_mag (5 leituras)": tabela["std_mag"],
    "std_mag (100, ESP32)": esp32["std_mag"],
})
comparacao

Com 5 leituras a conta chega perto. Com 100 o número fica mais **estável**, principalmente no
chacoalho, em que cada leitura é muito diferente da outra. É por isso que o ESP32 resume
uma janela de 100 leituras, e não de 5.

## 5) Divide por *n* ou por *n − 1*?

In [ ]:
import numpy as np

v = [3.945, 1.688, -0.468, -1.398, 1.315]   # az da anomalia

print("divide por n     statistics.pstdev(v) :", round(statistics.pstdev(v), 3))
print("divide por n     np.std(v)            :", round(float(np.std(v)), 3))
print("divide por n - 1 statistics.stdev(v)  :", round(statistics.stdev(v), 3))
print("divide por n - 1 pd.Series(v).std()   :", round(pd.Series(v).std(), 3))

`statistics.pstdev` e o **numpy** dividem por *n*, como o ESP32. `statistics.stdev` e o
**pandas** dividem por *n − 1* (o desvio "amostral"). Com 100 leituras a diferença é
pequena, mas a conta precisa ser **a mesma no treino e na inferência**. Por isso, aqui e no
firmware, divide-se por *n*.